In [1]:
# Core libraries
import os
import sys
import json
import copy
import pickle
import warnings
import gc
from datetime import datetime
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from collections import defaultdict, Counter
import random
import time
from tqdm import tqdm

# Image processing
import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau, StepLR
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import timm

# Metrics and visualization
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# Kaggle datasets
import kagglehub

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Set device for PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Output directory for training results
OUTPUT_DIR = Path('training_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ All libraries imported succeful")

/home/sufi/miniconda3/envs/organized/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
✅ All libraries imported succeful


In [2]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 77 — GLOBALS (run FIRST, always)                              ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, gc, torch, numpy as np, pandas as pd
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Semantic defect groups ─────────────────────────────────────────────
SEMANTIC_GROUPS = [
    'contamination', 'cut', 'deformation', 'fracture',
    'hole_void', 'minor_defect', 'scratch', 'surface_quality'
]

NUM_DEFECT_TYPES  = len(SEMANTIC_GROUPS)   # 8
NUM_PRODUCT_TYPES = 17

DEFECT_TYPE_TO_IDX = {name: idx for idx, name in enumerate(SEMANTIC_GROUPS)}
IDX_TO_DEFECT_TYPE = {idx: name for idx, name in enumerate(SEMANTIC_GROUPS)}

# ── Raw label → semantic group mapping ────────────────────────────────
DEFECT_GROUP_MAP = {
    # raw dataset labels
    'crack':               'fracture',
    'fracture':            'fracture',
    'faulty_imprint':      'fracture',
    'hole':                'hole_void',
    'void':                'hole_void',
    'pit':                 'hole_void',
    'blowhole':            'hole_void',
    'scratch':             'scratch',
    'score':               'scratch',
    'stain':               'surface_quality',
    'color':               'surface_quality',
    'rough':               'surface_quality',
    'uneven':              'surface_quality',
    'inclusion':           'surface_quality',
    'discolor':            'surface_quality',
    'pilling':             'surface_quality',
    'bent':                'deformation',
    'bent_lead':           'deformation',
    'squeeze':             'deformation',
    'deformation':         'deformation',
    'contamination':       'contamination',
    'glue':                'contamination',
    'oil':                 'contamination',
    'glue_strip':          'contamination',
    'liquid':              'contamination',
    'metal_contamination': 'contamination',
    'missing':             'minor_defect',
    'misplaced':           'minor_defect',
    'flip':                'minor_defect',
    'missing_hole':        'minor_defect',
    'thread':              'minor_defect',
    'cable_swap':          'minor_defect',
    'combined':            'minor_defect',
    'cut':                 'cut',
    # ── direct semantic group name passthroughs ────────────────────
    'hole_void':           'hole_void',
    'surface_quality':     'surface_quality',
    'minor_defect':        'minor_defect',
}

# ── Product type mapping ───────────────────────────────────────────────
PRODUCT_TYPE_TO_IDX = {}
IDX_TO_PRODUCT_TYPE = {}

print(f"✅ CELL 77 COMPLETE")
print(f"   NUM_DEFECT_TYPES  : {NUM_DEFECT_TYPES}")
print(f"   NUM_PRODUCT_TYPES : {NUM_PRODUCT_TYPES}")
print(f"   Semantic groups   : {SEMANTIC_GROUPS}")

Device: cuda
✅ CELL 77 COMPLETE
   NUM_DEFECT_TYPES  : 8
   NUM_PRODUCT_TYPES : 17
   Semantic groups   : ['contamination', 'cut', 'deformation', 'fracture', 'hole_void', 'minor_defect', 'scratch', 'surface_quality']


In [3]:
import torch
import torch.nn as nn
import torchvision.models as models

class CoordAttention(nn.Module):
    def __init__(self, channels, reduction=32):
        super().__init__()
        mid = max(8, channels // reduction)
        self.conv1  = nn.Conv2d(channels, mid, 1, bias=False)
        self.bn1    = nn.BatchNorm2d(mid)
        self.act    = nn.ReLU()
        self.conv_h = nn.Conv2d(mid, channels, 1, bias=False)
        self.conv_w = nn.Conv2d(mid, channels, 1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y   = torch.cat([x_h, x_w], dim=2)
        y   = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [H, W], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        return x * torch.sigmoid(self.conv_h(x_h)) * torch.sigmoid(self.conv_w(x_w))


class EdgeNetV2(nn.Module):
    def __init__(self, num_defect=8, num_product=17):
        super().__init__()
        base = models.mobilenet_v3_large(
            weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)

        self.early_features = base.features[:7]
        self.coord_att1     = CoordAttention(40)
        self.late_features  = base.features[7:]
        self.coord_att2     = CoordAttention(960)
        self.pool           = nn.AdaptiveAvgPool2d(1)

        self.shared = nn.Sequential(
            nn.Linear(960, 512),   # shared.0
            nn.Hardswish(),
            nn.Dropout(0.2),
        )

        # ✅ FIXED: 512 → 128 → 1   (confirmed from checkpoint shape [128,512])
        self.binary_head = nn.Sequential(
            nn.Linear(512, 128),   # binary_head.0
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),     # binary_head.3
        )

        # defect_head: 512 → 256 → num_defect
        self.defect_head = nn.Sequential(
            nn.Linear(512, 256),   # defect_head.0
            nn.Hardswish(),
            nn.Dropout(0.3),
            nn.Linear(256, num_defect),  # defect_head.3
        )

        # product_head: 512 → 256 → num_product
        self.product_head = nn.Sequential(
            nn.Linear(512, 256),   # product_head.0
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(256, num_product),  # product_head.3
        )

    def freeze_backbone(self, freeze=True):
        for p in self.early_features.parameters():
            p.requires_grad = not freeze
        for p in self.late_features.parameters():
            p.requires_grad = not freeze

    def count_params(self, trainable_only=False):
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

    def forward(self, x):
        x = self.early_features(x)
        x = self.coord_att1(x)
        x = self.late_features(x)
        x = self.coord_att2(x)
        x = self.pool(x).flatten(1)
        x = self.shared(x)
        return self.binary_head(x), self.defect_head(x), self.product_head(x)


# ── Build and load ────────────────────────────────────────────────────
model = EdgeNetV2(
    num_defect  = NUM_DEFECT_TYPES,
    num_product = NUM_PRODUCT_TYPES,
).to(device)

CKPT = '/home/sufi/training_results/models/EdgeNet_V2.pth'
ck   = torch.load(CKPT, map_location=device, weights_only=False)
missing, unexpected = model.load_state_dict(ck['model_state_dict'], strict=False)

print(f"Total params   : {model.count_params()/1e6:.2f}M")
print(f"Missing keys   : {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")

if len(missing) == 0 and len(unexpected) == 0:
    print("✅ PERFECT MATCH")
    model.eval()
    vm = ck['val_metrics']
    print(f"Epoch {ck['epoch']} | Binary={vm['binary_acc']:.1f}% "
          f"Defect={vm['defect_acc']:.1f}% F1={vm['defect_f1_macro']:.3f} "
          f"Product={vm['product_acc']:.1f}%")
else:
    # Auto-print shape mismatches for debugging
    s_sd = ck['model_state_dict']
    print("⚠️  Still mismatched — shape comparison:")
    for k in list(missing)[:5]:
        ck_shape = s_sd[k].shape if k in s_sd else "NOT IN CKPT"
        model_shape = dict(model.named_parameters()).get(k, None)
        model_shape = model_shape.shape if model_shape is not None else "NOT IN MODEL"
        print(f"  {k}: ckpt={ck_shape}  model={model_shape}")

print("\n✅ CELL 1 COMPLETE")

Total params   : 3.89M
Missing keys   : 0
Unexpected keys: 0
✅ PERFECT MATCH
Epoch 87 | Binary=98.1% Defect=86.8% F1=0.844 Product=99.9%

✅ CELL 1 COMPLETE


In [4]:
# ── AUG FIX — FIXED VERSION ───────────────────────────────────────────

CSV_PATH = '/home/sufi/merged_dataset_metadata_augmented.csv'
df_3head = pd.read_csv(CSV_PATH)

# ── Safe rename — only rename if old names exist ─────────────────────
if 'path' in df_3head.columns:
    df_3head = df_3head.rename(columns={'path': 'image_path'})
if 'category' in df_3head.columns:
    df_3head = df_3head.rename(columns={'category': 'product_type'})

print(f"Loaded CSV : {len(df_3head):,} rows")
print(f"Columns    : {list(df_3head.columns)}")

# ── Binary label ──────────────────────────────────────────────────────
def compute_binary(v):
    if pd.isna(v): return 0
    return 0 if str(v).strip().lower() in (
        '0','good','normal','ok','casting_ok') else 1

df_3head['binary_label'] = df_3head['label'].apply(compute_binary)
print(f"\n   binary_label : {df_3head['binary_label'].value_counts().to_dict()}")

# ── Defect label ──────────────────────────────────────────────────────
def remap_defect(raw):
    if pd.isna(raw): return -1
    s   = str(raw).strip().lower()
    if s in ('good','normal','casting_ok',''): return -1
    sem = DEFECT_GROUP_MAP.get(s, None)
    if sem is None and s in SEMANTIC_GROUPS: sem = s
    if sem is None:
        for k, v in DEFECT_GROUP_MAP.items():
            if k in s: sem = v; break
    return -1 if sem is None else DEFECT_TYPE_TO_IDX.get(sem, -1)

df_3head['defect_type_label'] = df_3head['defect_type'].apply(remap_defect)
df_3head.loc[df_3head['binary_label'] == 0, 'defect_type_label'] = -1

# ── Product label — use explicit loop to avoid numpy.matrix error ─────
all_products        = sorted(df_3head['product_type'].dropna().unique().tolist())
PRODUCT_TYPE_TO_IDX = {p: i for i, p in enumerate(all_products)}
IDX_TO_PRODUCT_TYPE = {i: p for i, p in enumerate(all_products)}
NUM_PRODUCT_TYPES   = len(all_products)

# Apply manually — avoids pandas .map() numpy.matrix bug
prod_labels = []
for v in df_3head['product_type']:
    if pd.isna(v):
        prod_labels.append(0)
    else:
        prod_labels.append(PRODUCT_TYPE_TO_IDX.get(str(v), 0))
df_3head['product_type_label'] = prod_labels

# ── Split ─────────────────────────────────────────────────────────────
train_df = df_3head[df_3head['split'] == 'train'].reset_index(drop=True)
val_df   = df_3head[df_3head['split'] == 'val'].reset_index(drop=True)
test_df  = df_3head[df_3head['split'] == 'test'].reset_index(drop=True)

unique = sorted(df_3head[
    df_3head['defect_type_label'] != -1
]['defect_type_label'].unique().tolist())

print(f"\n   Products       : {NUM_PRODUCT_TYPES} → {all_products}")
print(f"   Defect labels  : {unique}")
print(f"   Train          : {len(train_df):,}")
print(f"   Val            : {len(val_df):,}")
print(f"   Test           : {len(test_df):,}")

print(f"\n📊 Val samples per defect class:")
for i, name in enumerate(SEMANTIC_GROUPS):
    n = (val_df['defect_type_label'] == i).sum()
    flag = '  ⚠️ LOW' if n < 15 else ''
    print(f"   [{i}] {name:<20} val={n:>4}{flag}")

print("\n✅ AUG FIX COMPLETE")

Loaded CSV : 21,344 rows
Columns    : ['image_path', 'dataset', 'product_type', 'defect_type', 'label', 'split', 'defect_group', 'defect_type_label', 'binary_label']

   binary_label : {1: 11688, 0: 9656}

   Products       : 17 → ['bottle', 'cable', 'capsule', 'carpet', 'casting_product', 'grid', 'hazelnut', 'leather', 'magnetic_tile', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']
   Defect labels  : [0, 1, 2, 3, 4, 5, 6, 7]
   Train          : 16,337
   Val            : 3,339
   Test           : 1,668

📊 Val samples per defect class:
   [0] contamination        val=  33
   [1] cut                  val=  17
   [2] deformation          val=  15
   [3] fracture             val=  43
   [4] hole_void            val=  54
   [5] minor_defect         val=  42
   [6] scratch              val=  29
   [7] surface_quality      val=  63

✅ AUG FIX COMPLETE


In [5]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 78 — DATASET                                                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

import torch
from torch.utils.data import Dataset, DataLoader
import cv2
from PIL import Image
import torchvision.transforms as transforms

class ThreeHeadDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── load image ───────────────────────────────────────────
        img_path = row['image_path']
        img      = cv2.imread(img_path)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)

        if self.transform:
            img = self.transform(img)

        binary_label  = int(row['binary_label'])
        defect_label  = int(row['defect_type_label'])
        product_label = int(row['product_type_label'])

        return img, binary_label, defect_label, product_label, img_path

# ── ADD THIS TO THE BOTTOM OF CELL 78 (after ThreeHeadDataset class) ──

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3, 0.3, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_3head_ds = ThreeHeadDataset(train_df, transform=train_transform)
val_3head_ds   = ThreeHeadDataset(val_df,   transform=val_transform)
test_3head_ds  = ThreeHeadDataset(test_df,  transform=val_transform)

print(f"✅ CELL 78 COMPLETE")
print(f"   train_3head_ds : {len(train_3head_ds):,} samples")
print(f"   val_3head_ds   : {len(val_3head_ds):,} samples")
print(f"   test_3head_ds  : {len(test_3head_ds):,} samples")

print("✅ CELL 78 COMPLETE — ThreeHeadDataset defined")

✅ CELL 78 COMPLETE
   train_3head_ds : 16,337 samples
   val_3head_ds   : 3,339 samples
   test_3head_ds  : 1,668 samples
✅ CELL 78 COMPLETE — ThreeHeadDataset defined


In [6]:
# ════════════════════════════════════════════════════════════════════════
# CELL 78C — CUTMIX DATALOADER (replaces Cell 78B)
# Run AFTER Cell 78 (which defines ThreeHeadDataset)
# ════════════════════════════════════════════════════════════════════════
 
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
 
# ── Weighted sampler (same as Cell 78B) ───────────────────────────────
def make_weighted_sampler(df):
    class_counts = np.ones(NUM_DEFECT_TYPES)
    for i in range(NUM_DEFECT_TYPES):
        class_counts[i] = max(1, (df['defect_type_label'] == i).sum())
    cw = 1.0 / class_counts
    cw = cw / cw.sum()
    weights = []
    for _, row in df.iterrows():
        lbl = int(row['defect_type_label'])
        weights.append(0.3 if lbl == -1 else float(cw[lbl]))
    weights = torch.tensor(weights, dtype=torch.float32)
    return WeightedRandomSampler(weights, len(weights), replacement=True)
 
sampler = make_weighted_sampler(train_df)
 
# ── CutMix collate function ────────────────────────────────────────────
def cutmix_collate(batch):
    """
    Standard collate + CutMix applied only to defect-labeled samples.
    Binary and product labels unchanged (CutMix only on defect head).
    """
    imgs, bin_l, def_l, prod_l, paths = zip(*batch)
    imgs   = torch.stack(imgs)
    bin_l  = torch.tensor(bin_l,  dtype=torch.long)
    def_l  = torch.tensor(def_l,  dtype=torch.long)
    prod_l = torch.tensor(prod_l, dtype=torch.long)
 
    # Only apply CutMix to samples with known defect labels
    known = (def_l != -1).nonzero(as_tuple=True)[0]
 
    if len(known) >= 4 and torch.rand(1).item() < 0.5:
        # Sample random lambda from Beta(1.0, 1.0)
        lam   = float(np.random.beta(1.0, 1.0))
        lam   = max(0.3, min(0.7, lam))   # constrain so neither label dominates
 
        # Shuffle known indices to get pairs
        perm  = known[torch.randperm(len(known))]
 
        B, C, H, W = imgs[known].shape
        # Random bounding box
        cut_ratio = np.sqrt(1 - lam)
        cut_h     = int(H * cut_ratio)
        cut_w     = int(W * cut_ratio)
        cx        = np.random.randint(W)
        cy        = np.random.randint(H)
        x1        = max(0, cx - cut_w // 2)
        x2        = min(W, cx + cut_w // 2)
        y1        = max(0, cy - cut_h // 2)
        y2        = min(H, cy + cut_h // 2)
 
        # Apply cut and paste
        imgs_copy = imgs.clone()
        imgs_copy[known, :, y1:y2, x1:x2] = imgs[perm, :, y1:y2, x1:x2]
 
        # Adjust lambda to actual area
        lam_actual = 1 - (x2 - x1) * (y2 - y1) / (H * W)
 
        # Soft labels for defect head only
        # Store both original and mixed labels; loss will handle soft targets
        # We use a simple approach: randomly assign label based on lambda
        # (avoids changing loss function signature)
        # For samples where lam > 0.5 keep original label, else swap
        if lam_actual >= 0.5:
            imgs = imgs_copy
        else:
            imgs = imgs_copy
            def_l_new = def_l.clone()
            def_l_new[known] = def_l[perm]
            def_l = def_l_new
 
    return imgs, bin_l, def_l, prod_l, list(paths)
 
 
# ── Build dataloaders ──────────────────────────────────────────────────
BATCH_SIZE  = 32
NUM_WORKERS = 4
 
train_3head_loader = DataLoader(
    train_3head_ds,
    batch_size  = BATCH_SIZE,
    sampler     = sampler,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
    drop_last   = True,
    collate_fn  = cutmix_collate,   # ← CutMix applied here
)
 
val_3head_loader = DataLoader(
    val_3head_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
    # No CutMix on val — clean evaluation
)
 
print(f"✅ CELL 78C COMPLETE — CutMix DataLoader")
print(f"   Train batches : {len(train_3head_loader)}")
print(f"   Val batches   : {len(val_3head_loader)}")
print(f"   CutMix        : ✅ applied in collate (p=0.5 per batch)")
print(f"   Weighted sampler: ✅ active")

✅ CELL 78C COMPLETE — CutMix DataLoader
   Train batches : 510
   Val batches   : 105
   CutMix        : ✅ applied in collate (p=0.5 per batch)
   Weighted sampler: ✅ active


In [8]:
# ────────────────────────────────────────────────────────────────────────
# CELL KD1 — FIXED: Teacher matches exact checkpoint key structure
# ────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torchvision.models as models
from pathlib import Path

class EfficientNetTeacher(nn.Module):
    """
    Architecture matches exactly what was saved in efficientnet_b0_best.pth.
    Key names confirmed from checkpoint: features.*, shared.0, 
    binary_head.0/2, defect_head.0/3, product_head.0/2
    """
    def __init__(self, num_defect_classes=8, num_product_classes=17):
        super().__init__()
        base           = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features  = base.features        # ← must be 'features' not 'backbone'
        self.pool      = nn.AdaptiveAvgPool2d(1)

        # shared.0 = single Linear
        self.shared = nn.Sequential(
            nn.Linear(1280, 512),             # shared.0
            nn.SiLU(),
            nn.Dropout(0.3),
        )

        # binary_head: index 0=Linear, 1=ReLU/dropout, 2=Linear
        self.binary_head = nn.Sequential(
            nn.Linear(512, 128),              # binary_head.0
            nn.ReLU(),
            nn.Linear(128, 1),               # binary_head.2
        )

        # defect_head: index 0=Linear, 1,2=act/drop, 3=Linear
        self.defect_head = nn.Sequential(
            nn.Linear(512, 256),              # defect_head.0
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_defect_classes), # defect_head.3
        )

        # product_head: index 0=Linear, 1=act/drop, 2=Linear
        self.product_head = nn.Sequential(
            nn.Linear(512, 256),              # product_head.0
            nn.SiLU(),
            nn.Linear(256, num_product_classes), # product_head.2
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        s = self.shared(x)
        return self.binary_head(s), self.defect_head(s), self.product_head(s)


# ── Load teacher ──────────────────────────────────────────────────────
TEACHER_CKPT = Path('/home/sufi/training_results/baseline_models/efficientnet_b0_best.pth')

teacher = EfficientNetTeacher(
    num_defect_classes  = NUM_DEFECT_TYPES,
    num_product_classes = NUM_PRODUCT_TYPES,
).to(device)

ck = torch.load(TEACHER_CKPT, map_location=device, weights_only=False)
# checkpoint IS the raw state dict — load directly
teacher.load_state_dict(ck, strict=False)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

params_t = sum(p.numel() for p in teacher.parameters()) / 1e6
print(f"✅ CELL KD1 COMPLETE — Teacher loaded")
print(f"   Params  : {params_t:.2f}M  (frozen)")

# ── Verify ────────────────────────────────────────────────────────────
with torch.no_grad():
    dummy    = torch.randn(2, 3, 224, 224).to(device)
    tb, td, tp = teacher(dummy)
    print(f"   Outputs : bin={tb.shape} def={td.shape} prod={tp.shape}  ✅")

✅ CELL KD1 COMPLETE — Teacher loaded
   Params  : 5.00M  (frozen)
   Outputs : bin=torch.Size([2, 1]) def=torch.Size([2, 8]) prod=torch.Size([2, 17])  ✅


In [9]:
# ======================================================================
# KD_CELL_1 — INSPECT BOTH CHECKPOINTS (verify before building anything)
# ======================================================================
import torch
from pathlib import Path
 
TEACHER_PATH = Path('/home/sufi/training_results/baseline_models/efficientnet_b0_best.pth')
STUDENT_PATH = Path('/home/sufi/training_results/models/EdgeNet_V2.pth')
 
print("=" * 60)
print("CHECKPOINT INSPECTION")
print("=" * 60)
 
# --- Teacher ---
ck_t = torch.load(TEACHER_PATH, map_location='cpu', weights_only=False)
# From your error log we KNOW this is a raw state dict
# Keys confirmed: features.0.0.weight ... shared.0.weight ...
# binary_head.0.weight/bias, binary_head.2.weight/bias
# defect_head.0.weight/bias, defect_head.3.weight/bias
# product_head.0.weight/bias, product_head.2.weight/bias
t_sd   = ck_t  # raw state dict — no wrapper
t_keys = list(t_sd.keys())
print(f"\n[TEACHER] {TEACHER_PATH.name}")
print(f"  Type        : raw state dict")
print(f"  Total keys  : {len(t_keys)}")
print(f"  First 3     : {t_keys[:3]}")
print(f"  Last  3     : {t_keys[-3:]}")
print(f"  shared shape: {t_sd['shared.0.weight'].shape}")     # should be [512,1280]
print(f"  binary shape: {t_sd['binary_head.0.weight'].shape}") # [128,512]
print(f"  defect shape: {t_sd['defect_head.0.weight'].shape}") # [256,512]
print(f"  product shp : {t_sd['product_head.0.weight'].shape}")# [256,512]
 
# --- Student ---
ck_s   = torch.load(STUDENT_PATH, map_location='cpu', weights_only=False)
s_sd   = ck_s['model_state_dict']
s_keys = list(s_sd.keys())
top    = sorted(set(k.split('.')[0] for k in s_keys))
print(f"\n[STUDENT] {STUDENT_PATH.name}")
print(f"  Epoch       : {ck_s['epoch']}")
print(f"  Val score   : {ck_s['val_score']:.2f}")
print(f"  Total keys  : {len(s_keys)}")
print(f"  Top-level   : {top}")
vm = ck_s.get('val_metrics', {})
print(f"  Binary acc  : {vm.get('binary_acc', 0):.2f}%")
print(f"  Defect acc  : {vm.get('defect_acc', 0):.2f}%")
print(f"  F1 macro    : {vm.get('defect_f1_macro', 0):.3f}")
print(f"  Product acc : {vm.get('product_acc', 0):.2f}%")
 
print("\n✅ KD_CELL_1 DONE — both checkpoints verified")
 
 

CHECKPOINT INSPECTION

[TEACHER] efficientnet_b0_best.pth
  Type        : raw state dict
  Total keys  : 372
  First 3     : ['features.0.0.weight', 'features.0.1.weight', 'features.0.1.bias']
  Last  3     : ['product_head.0.bias', 'product_head.2.weight', 'product_head.2.bias']
  shared shape: torch.Size([512, 1280])
  binary shape: torch.Size([128, 512])
  defect shape: torch.Size([256, 512])
  product shp : torch.Size([256, 512])

[STUDENT] EdgeNet_V2.pth
  Epoch       : 87
  Val score   : 92.48
  Total keys  : 338
  Top-level   : ['binary_head', 'coord_att1', 'coord_att2', 'defect_head', 'early_features', 'late_features', 'product_head', 'shared']
  Binary acc  : 98.14%
  Defect acc  : 86.82%
  F1 macro    : 0.844
  Product acc : 99.94%

✅ KD_CELL_1 DONE — both checkpoints verified


In [10]:
# ======================================================================
# KD_CELL_2 — BUILD TEACHER + VERIFY ACCURACY BEFORE TRAINING
# ======================================================================
import torch.nn as nn
import torchvision.models as models
 
# Architecture built to EXACTLY match confirmed checkpoint keys:
#   features.*           → self.features  (EfficientNet backbone)
#   shared.0             → Linear(1280, 512)
#   binary_head.0 / .2   → Linear(512,128) → Linear(128,1)
#   defect_head.0 / .3   → Linear(512,256) → Linear(256,8)
#   product_head.0 / .2  → Linear(512,256) → Linear(256,17)
 
class EfficientNetTeacher(nn.Module):
    def __init__(self, num_defect=8, num_product=17):
        super().__init__()
        base          = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = base.features          # ← key prefix MUST be 'features'
        self.pool     = nn.AdaptiveAvgPool2d(1)
 
        self.shared = nn.Sequential(           # shared.0 = Linear(1280,512)
            nn.Linear(1280, 512),
            nn.SiLU(),
            nn.Dropout(0.3),
        )
        self.binary_head = nn.Sequential(      # .0=Linear(512,128)  .2=Linear(128,1)
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )
        self.defect_head = nn.Sequential(      # .0=Linear(512,256)  .3=Linear(256,8)
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_defect),
        )
        self.product_head = nn.Sequential(     # .0=Linear(512,256)  .2=Linear(256,17)
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Linear(256, num_product),
        )
 
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        s = self.shared(x)
        return self.binary_head(s), self.defect_head(s), self.product_head(s)
 
 
teacher = EfficientNetTeacher(
    num_defect  = NUM_DEFECT_TYPES,
    num_product = NUM_PRODUCT_TYPES,
).to(device)
 
# Load — checkpoint is raw state dict (no wrapper)
missing, unexpected = teacher.load_state_dict(ck, strict=False)
 
# Freeze completely
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
 
params_t = sum(p.numel() for p in teacher.parameters()) / 1e6
 
# --- Run on full val set to print real accuracy ---
teacher.eval()
tb_c = td_c = tp_c = td_tot = t_tot = 0
with torch.no_grad():
    for batch in val_3head_loader:
        imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
        imgs = imgs.to(device)
        bin_lbl  = bin_lbl.to(device)
        def_lbl  = def_lbl.to(device)
        prod_lbl = prod_lbl.to(device)
        tb2, td2, tp2 = teacher(imgs)
        tb_c  += ((torch.sigmoid(tb2.squeeze()) > 0.5).long() == bin_lbl).sum().item()
        known  = def_lbl != -1
        if known.sum() > 0:
            td_c   += (td2[known].argmax(1) == def_lbl[known]).sum().item()
            td_tot += known.sum().item()
        tp_c  += (tp2.argmax(1) == prod_lbl).sum().item()
        t_tot += imgs.size(0)
 
t_bin_acc  = 100. * tb_c / max(t_tot, 1)
t_def_acc  = 100. * td_c / max(td_tot, 1)
t_prod_acc = 100. * tp_c / max(t_tot, 1)
 
print("=" * 60)
print("TEACHER VERIFICATION")
print("=" * 60)
print(f"  Architecture : EfficientNet-B0")
print(f"  Parameters   : {params_t:.2f}M  (all frozen)")
print(f"  Missing keys : {len(missing)}   Unexpected: {len(unexpected)}")
print(f"  --- Full Val Set Accuracy ---")
print(f"  Binary Acc   : {t_bin_acc:.2f}%")
print(f"  Defect Acc   : {t_def_acc:.2f}%")
print(f"  Product Acc  : {t_prod_acc:.2f}%")
print("=" * 60)
print("\n✅ KD_CELL_2 DONE — Teacher verified ✅")
 
 


TEACHER VERIFICATION
  Architecture : EfficientNet-B0
  Parameters   : 5.00M  (all frozen)
  Missing keys : 0   Unexpected: 0
  --- Full Val Set Accuracy ---
  Binary Acc   : 99.01%
  Defect Acc   : 92.57%
  Product Acc  : 99.79%

✅ KD_CELL_2 DONE — Teacher verified ✅


In [11]:
# ======================================================================
# KD_CELL_3 — LOAD STUDENT + VERIFY ACCURACY BEFORE TRAINING
# ======================================================================
# model is already defined from Cell 1 (EdgeNet-V2)
# Just load the best checkpoint weights
 
model.load_state_dict(ck_s['model_state_dict'])
params_s = sum(p.numel() for p in model.parameters()) / 1e6
 
# --- Run on full val set ---
model.eval()
sb_c = sd_c = sp_c = sd_tot = s_tot = 0
with torch.no_grad():
    for batch in val_3head_loader:
        imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
        imgs = imgs.to(device)
        bin_lbl  = bin_lbl.to(device)
        def_lbl  = def_lbl.to(device)
        prod_lbl = prod_lbl.to(device)
        sb2, sd2, sp2 = model(imgs)
        sb_c  += ((torch.sigmoid(sb2.squeeze()) > 0.5).long() == bin_lbl).sum().item()
        known  = def_lbl != -1
        if known.sum() > 0:
            sd_c   += (sd2[known].argmax(1) == def_lbl[known]).sum().item()
            sd_tot += known.sum().item()
        sp_c  += (sp2.argmax(1) == prod_lbl).sum().item()
        s_tot += imgs.size(0)
 
s_bin_acc  = 100. * sb_c / max(s_tot, 1)
s_def_acc  = 100. * sd_c / max(sd_tot, 1)
s_prod_acc = 100. * sp_c / max(s_tot, 1)
 
print("=" * 60)
print("STUDENT VERIFICATION")
print("=" * 60)
print(f"  Architecture : EdgeNet-V2")
print(f"  Parameters   : {params_s:.2f}M")
print(f"  Loaded from  : EdgeNet_V2.pth  (epoch {ck_s['epoch']})")
print(f"  --- Full Val Set Accuracy ---")
print(f"  Binary Acc   : {s_bin_acc:.2f}%")
print(f"  Defect Acc   : {s_def_acc:.2f}%")
print(f"  Product Acc  : {s_prod_acc:.2f}%")
print(f"\n  TEACHER vs STUDENT (before KD):")
print(f"  {'Model':<22} {'Params':>7}  {'Binary':>8}  {'Defect':>8}  {'Product':>9}")
print(f"  {'-'*58}")
print(f"  {'EfficientNet-B0':<22} {params_t:>6.2f}M  "
      f"{t_bin_acc:>7.2f}%  {t_def_acc:>7.2f}%  {t_prod_acc:>8.2f}%  ← Teacher")
print(f"  {'EdgeNet-V2':<22} {params_s:>6.2f}M  "
      f"{s_bin_acc:>7.2f}%  {s_def_acc:>7.2f}%  {s_prod_acc:>8.2f}%  ← Student")
print("=" * 60)
print("\n✅ KD_CELL_3 DONE — Student verified ✅")


STUDENT VERIFICATION
  Architecture : EdgeNet-V2
  Parameters   : 3.89M
  Loaded from  : EdgeNet_V2.pth  (epoch 87)
  --- Full Val Set Accuracy ---
  Binary Acc   : 97.93%
  Defect Acc   : 83.45%
  Product Acc  : 99.76%

  TEACHER vs STUDENT (before KD):
  Model                   Params    Binary    Defect    Product
  ----------------------------------------------------------
  EfficientNet-B0          5.00M    99.01%    92.57%     99.79%  ← Teacher
  EdgeNet-V2               3.89M    97.93%    83.45%     99.76%  ← Student

✅ KD_CELL_3 DONE — Student verified ✅


In [14]:
# ======================================================================
# KD_CELL_3B — BUILD criterion_dict  (run before KD_CELL_4)
# ======================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Per-class focal loss ───────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma_per_class, weight=None, ignore_index=-1):
        super().__init__()
        self.gamma        = torch.tensor(gamma_per_class, dtype=torch.float32)
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        valid   = targets != self.ignore_index
        if valid.sum() == 0:
            return logits.sum() * 0.0
        logits  = logits[valid]
        targets = targets[valid]
        gamma   = self.gamma.to(logits.device)
        ce      = F.cross_entropy(logits, targets,
                                  weight=self.weight,
                                  reduction='none')
        pt      = torch.exp(-ce)
        g       = gamma[targets]
        return ((1 - pt) ** g * ce).mean()


# ── Homoscedastic uncertainty multi-task weighting ────────────────────
class HomoscedasticLoss(nn.Module):
    """
    Learnable log-variance per task.
    loss_i = exp(-log_var_i) * task_loss_i + log_var_i
    """
    def __init__(self, n_tasks=3):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(n_tasks))

    def weights(self):
        return torch.exp(-self.log_vars).detach()

    def set_epoch(self, epoch):
        pass   # no-op; kept for API compatibility

    def state_dict(self, **kwargs):
        return super().state_dict(**kwargs)

    def forward(self, losses):
        w = torch.exp(-self.log_vars)
        return sum(w[i] * losses[i] + self.log_vars[i]
                   for i in range(len(losses)))


# ── Compute class weights from training set ───────────────────────────
def compute_defect_weights(train_df, num_classes, device):
    counts = torch.zeros(num_classes)
    for i in range(num_classes):
        counts[i] = max(1, (train_df['defect_type_label'] == i).sum())
    w = 1.0 / counts
    return (w / w.sum() * num_classes).to(device)

defect_weights = compute_defect_weights(train_df, NUM_DEFECT_TYPES, device)

# Gamma per defect class (thesis values)
# contamination=2.0, cut=2.0, deformation=3.5, fracture=2.0,
# hole_void=2.0, minor_defect=2.0, scratch=4.0, surface_quality=2.0
GAMMA_PER_CLASS = [2.0, 2.0, 3.5, 2.0, 2.0, 2.0, 4.0, 2.0]

# Positive-class weight for binary head (defective:normal ratio)
n_defective = (train_df['binary_label'] == 1).sum()
n_normal    = (train_df['binary_label'] == 0).sum()
pos_weight  = torch.tensor([n_normal / max(n_defective, 1)],
                            dtype=torch.float32).to(device)

criterion_dict = {
    'binary':    nn.BCEWithLogitsLoss(pos_weight=pos_weight),
    'defect':    FocalLoss(gamma_per_class=GAMMA_PER_CLASS,
                           weight=defect_weights,
                           ignore_index=-1),
    'product':   nn.CrossEntropyLoss(),
    'multitask': HomoscedasticLoss(n_tasks=3).to(device),
}

print("✅ KD_CELL_3B COMPLETE — criterion_dict built")
print(f"   Keys          : {list(criterion_dict.keys())}")
print(f"   pos_weight    : {pos_weight.item():.3f}")
print(f"   defect_weights: {[f'{w:.3f}' for w in defect_weights.cpu()]}")
print(f"   gamma/class   : {GAMMA_PER_CLASS}")
tw = criterion_dict['multitask'].weights()
print(f"   task weights  : {tw}")

✅ KD_CELL_3B COMPLETE — criterion_dict built
   Keys          : ['binary', 'defect', 'product', 'multitask']
   pos_weight    : 0.706
   defect_weights: ['0.601', '0.893', '0.459', '0.738', '1.039', '2.756', '0.512', '1.002']
   gamma/class   : [2.0, 2.0, 3.5, 2.0, 2.0, 2.0, 4.0, 2.0]
   task weights  : tensor([1., 1., 1.], device='cuda:0')


In [16]:
# ─────────────────────────────────────────────────────────────────────
# KD_CELL_4 — KD LOSS
# ─────────────────────────────────────────────────────────────────────
import torch.nn.functional as F
 
class KDLoss(nn.Module):
    def __init__(self, T_def=4.0, T_bin=2.0, T_prod=3.0, alpha=0.7):
        super().__init__()
        self.T_def=T_def; self.T_bin=T_bin; self.T_prod=T_prod; self.alpha=alpha
 
    def set_alpha(self, a): self.alpha = a
 
    def _kl(self, s, t, T):
        return F.kl_div(F.log_softmax(s/T,dim=1),
                        F.softmax(t/T,dim=1),
                        reduction='batchmean') * (T*T)
 
    def forward(self, s_bin, s_def, s_prod,
                      t_bin, t_def, t_prod,
                      bin_lbl, def_lbl, prod_lbl, criterion_dict):
        hard_bin  = criterion_dict['binary'](s_bin, bin_lbl.float())
        hard_def  = criterion_dict['defect'](s_def, def_lbl)
        hard_prod = criterion_dict['product'](s_prod, prod_lbl)
        s_b2 = torch.stack([-s_bin, s_bin], dim=1)
        t_b2 = torch.stack([-t_bin, t_bin], dim=1)
        soft_bin  = self._kl(s_b2, t_b2, self.T_bin)
        soft_def  = self._kl(s_def, t_def, self.T_def)
        soft_prod = self._kl(s_prod, t_prod, self.T_prod)
        a = self.alpha
        lb = a*hard_bin  + (1-a)*soft_bin
        ld = a*hard_def  + (1-a)*soft_def
        lp = a*hard_prod + (1-a)*soft_prod
        tw = criterion_dict['multitask'].weights()
        total = float(tw[0])*lb + float(tw[1])*ld + float(tw[2])*lp
        return total, lb, ld, lp
 
kd_loss = KDLoss(T_def=4.0, T_bin=2.0, T_prod=3.0, alpha=0.7)
 
with torch.no_grad():
    _img = torch.randn(4,3,224,224).to(device)
    _sb, _sd, _sp = model(_img)
    _tb, _td, _tp = teacher(_img)
    _sb=_sb.squeeze(); _tb=_tb.squeeze()
    _bl=torch.randint(0,2,(4,)).to(device)
    _dl=torch.randint(0,NUM_DEFECT_TYPES,(4,)).to(device)
    _pl=torch.randint(0,NUM_PRODUCT_TYPES,(4,)).to(device)
    tot,lb,ld,lp = kd_loss(_sb,_sd,_sp,_tb,_td,_tp,_bl,_dl,_pl,criterion_dict)
    print(f"  KD sanity: total={tot:.4f} bin={lb:.4f} def={ld:.4f} prod={lp:.4f}")
print("✅ KD_CELL_4 DONE")


  KD sanity: total=9.7394 bin=1.7977 def=1.5972 prod=6.3445
✅ KD_CELL_4 DONE


In [17]:
# ======================================================================
# KD_CELL_5 — KD TRAINING LOOP  (100 epochs)
# Phases : FROZEN (epochs 1–5) → FINETUNE (epochs 6–100)
# α      : 0.7 → 0.5 annealed over first 20 epochs
# Output : matches confirmed log format
# ======================================================================

from sklearn.metrics import precision_recall_fscore_support
from torch.cuda.amp import GradScaler, autocast
from pathlib import Path
import copy, torch
import numpy as np

KD_EPOCHS    = 100
FROZEN_PHASE = 5
ALPHA_START  = 0.7
ALPHA_END    = 0.5
ALPHA_ANNEAL = 20
KD_SAVE_PATH = Path('/home/sufi/training_results/models/EdgeNet_V2_KD.pth')

# ── Differential-LR optimizer factory ─────────────────────────────────
def make_kd_optimizer(model, bb_lr):
    bb_params    = list(model.early_features.parameters()) + \
                   list(model.late_features.parameters())
    coord_params = list(model.coord_att1.parameters()) + \
                   list(model.coord_att2.parameters())
    head_params  = list(model.shared.parameters()) + \
                   list(model.binary_head.parameters()) + \
                   list(model.defect_head.parameters()) + \
                   list(model.product_head.parameters())
    return torch.optim.AdamW([
        {'params': bb_params,    'lr': bb_lr},
        {'params': coord_params, 'lr': 1e-4},
        {'params': head_params,  'lr': 1e-4},
    ], weight_decay=1e-4)

# Start with backbone frozen
optimizer = make_kd_optimizer(model, bb_lr=0.0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=KD_EPOCHS, eta_min=1e-6)
scaler    = GradScaler()

# ── Alpha schedule ─────────────────────────────────────────────────────
def get_alpha(epoch):
    if epoch >= ALPHA_ANNEAL:
        return ALPHA_END
    t = epoch / ALPHA_ANNEAL
    return ALPHA_START - t * (ALPHA_START - ALPHA_END)

# ── Validation + per-class metrics ────────────────────────────────────
def evaluate_kd(model, loader):
    model.eval()
    all_pred, all_true = [], []
    bin_c = prod_c = tot = def_c = def_tot = 0
    with torch.no_grad():
        for batch in loader:
            imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
            imgs     = imgs.to(device)
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)
            sb, sd, sp = model(imgs)
            bin_c  += ((torch.sigmoid(sb.squeeze()) > 0.5).long()
                       == bin_lbl).sum().item()
            prod_c += (sp.argmax(1) == prod_lbl).sum().item()
            tot    += imgs.size(0)
            known   = def_lbl != -1
            if known.sum() > 0:
                preds = sd[known].argmax(1)
                def_c   += (preds == def_lbl[known]).sum().item()
                def_tot += known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())

    bin_acc  = 100. * bin_c  / max(tot,     1)
    def_acc  = 100. * def_c  / max(def_tot, 1)
    prod_acc = 100. * prod_c / max(tot,     1)

    if all_true:
        p, r, f, _ = precision_recall_fscore_support(
            all_true, all_pred,
            labels=list(range(NUM_DEFECT_TYPES)),
            zero_division=0)
        f1_macro = float(f.mean())
    else:
        p = r = f = np.zeros(NUM_DEFECT_TYPES)
        f1_macro = 0.0

    return bin_acc, def_acc, prod_acc, f1_macro, p, r, f

# ── Training ───────────────────────────────────────────────────────────
best_f1   = 0.0
phase     = 'FROZEN'
model.freeze_backbone(True)

for epoch in range(1, KD_EPOCHS + 1):

    # ── Phase switch at epoch FROZEN_PHASE+1 ──────────────────────────
    if epoch == FROZEN_PHASE + 1:
        model.freeze_backbone(False)
        optimizer = make_kd_optimizer(model, bb_lr=2e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max  = KD_EPOCHS - FROZEN_PHASE,
            eta_min= 1e-6)
        phase = 'FINETUNE'

    # ── Update alpha + task weights ───────────────────────────────────
    alpha = get_alpha(epoch - 1)
    kd_loss.set_alpha(alpha)
    criterion_dict['multitask'].set_epoch(epoch)

    # ── Train one epoch ───────────────────────────────────────────────
    model.train()
    teacher.eval()

    tr_bin_c = tr_def_c = tr_prod_c = tr_def_tot = tr_tot = 0
    tr_def_pred, tr_def_true = [], []

    for batch in train_3head_loader:
        imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
        imgs     = imgs.to(device)
        bin_lbl  = bin_lbl.to(device)
        def_lbl  = def_lbl.to(device)
        prod_lbl = prod_lbl.to(device)

        optimizer.zero_grad()

        with autocast():
            s_bin, s_def, s_prod = model(imgs)
            with torch.no_grad():
                t_bin, t_def, t_prod = teacher(imgs)

            s_bin_sq = s_bin.squeeze()
            t_bin_sq = t_bin.squeeze()
            known    = def_lbl != -1

            # ── Hard losses ───────────────────────────────────────────
            hard_bin  = criterion_dict['binary'](s_bin_sq, bin_lbl.float())
            hard_prod = criterion_dict['product'](s_prod, prod_lbl)

            if known.sum() > 0:
                hard_def = criterion_dict['defect'](
                    s_def[known], def_lbl[known])
                soft_def = kd_loss._kl(
                    s_def[known], t_def[known], kd_loss.T_def)
            else:
                hard_def = torch.tensor(0.0, device=device)
                soft_def = torch.tensor(0.0, device=device)

            # ── Soft losses ───────────────────────────────────────────
            s_b2      = torch.stack([-s_bin_sq, s_bin_sq], dim=1)
            t_b2      = torch.stack([-t_bin_sq, t_bin_sq], dim=1)
            soft_bin  = kd_loss._kl(s_b2,   t_b2,   kd_loss.T_bin)
            soft_prod = kd_loss._kl(s_prod, t_prod, kd_loss.T_prod)

            a  = kd_loss.alpha
            lb = a * hard_bin  + (1 - a) * soft_bin
            ld = a * hard_def  + (1 - a) * soft_def
            lp = a * hard_prod + (1 - a) * soft_prod

            tw         = criterion_dict['multitask'].weights()
            total_loss = float(tw[0])*lb + float(tw[1])*ld + float(tw[2])*lp

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        # ── Accumulate train metrics ──────────────────────────────────
        bs = imgs.size(0)
        tr_tot    += bs
        tr_bin_c  += ((torch.sigmoid(s_bin_sq.detach()) > 0.5).long()
                      == bin_lbl).sum().item()
        tr_prod_c += (s_prod.detach().argmax(1) == prod_lbl).sum().item()
        if known.sum() > 0:
            preds = s_def.detach()[known].argmax(1)
            tr_def_c   += (preds == def_lbl[known]).sum().item()
            tr_def_tot += known.sum().item()
            tr_def_pred.extend(preds.cpu().tolist())
            tr_def_true.extend(def_lbl[known].cpu().tolist())

    scheduler.step()

    # ── Train metrics ─────────────────────────────────────────────────
    t_bin_acc  = 100. * tr_bin_c  / max(tr_tot,     1)
    t_def_acc  = 100. * tr_def_c  / max(tr_def_tot, 1)
    t_prod_acc = 100. * tr_prod_c / max(tr_tot,     1)

    if tr_def_true:
        _, _, tf, _ = precision_recall_fscore_support(
            tr_def_true, tr_def_pred,
            labels=list(range(NUM_DEFECT_TYPES)),
            zero_division=0)
        t_f1 = float(tf.mean())
    else:
        t_f1 = 0.0

    # ── Validation metrics ────────────────────────────────────────────
    v_bin, v_def, v_prod, v_f1, vp, vr, vf = evaluate_kd(
        model, val_3head_loader)

    # ── Current LR + task weights ─────────────────────────────────────
    cur_lr = optimizer.param_groups[1]['lr']   # coord_att group
    tw     = criterion_dict['multitask'].weights()

    # ── Print — matches confirmed log format ──────────────────────────
    print(f"\nEpoch [{epoch}/{KD_EPOCHS}]  LR={cur_lr:.1e}  "
          f"TaskW=[bin={float(tw[0]):.2f} "
          f"def={float(tw[1]):.2f} "
          f"prod={float(tw[2]):.2f}]  [{phase}]")
    print()
    print(f"  Train → Bin={t_bin_acc:.1f}%  Def={t_def_acc:.1f}%  "
          f"F1={t_f1:.3f}  Prod={t_prod_acc:.1f}%")
    print(f"  Val   → Bin={v_bin:.1f}%  Def={v_def:.1f}%  "
          f"F1={v_f1:.3f}  Prod={v_prod:.1f}%")
    print()
    print(f"  {'Class':<22} {'P':>6}  {'R':>6}  {'F1':>6}")
    print(f"  {'-'*40}")
    for i, name in enumerate(SEMANTIC_GROUPS):
        print(f"  {name:<22} {vp[i]:>6.3f}  {vr[i]:>6.3f}  {vf[i]:>6.3f}")
    print(f"  {'-'*40}")
    print(f"  {'MACRO':<22} {float(np.mean(vp)):>6.3f}  "
          f"{float(np.mean(vr)):>6.3f}  {v_f1:>6.3f}")

    # ── Save best checkpoint ──────────────────────────────────────────
    if v_f1 > best_f1:
        best_f1 = v_f1
        torch.save({
            'epoch':               epoch,
            'model_state_dict':    model.state_dict(),
            'multitask_state':     criterion_dict['multitask'].state_dict()
                                   if hasattr(criterion_dict['multitask'],
                                              'state_dict') else None,
            'optimizer_state':     optimizer.state_dict(),
            'val_score':           v_f1,
            'val_metrics': {
                'binary_acc':       v_bin,
                'defect_acc':       v_def,
                'defect_f1_macro':  v_f1,
                'product_acc':      v_prod,
            },
            'defect_type_to_idx':  DEFECT_TYPE_TO_IDX,
            'product_type_to_idx': PRODUCT_TYPE_TO_IDX,
            'idx_to_defect_type':  IDX_TO_DEFECT_TYPE,
            'idx_to_product_type': IDX_TO_PRODUCT_TYPE,
            'kd_alpha':            alpha,
        }, KD_SAVE_PATH)
        print(f"\n  ✅ NEW BEST  F1={best_f1:.3f} → saved EdgeNet_V2_KD.pth")

# ── Summary ────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"KD TRAINING COMPLETE")
print(f"  Best Val F1 (macro) : {best_f1:.3f}")
print(f"  Saved to            : {KD_SAVE_PATH}")
print(f"{'='*60}")
print("✅ KD_CELL_5 DONE")


Epoch [1/100]  LR=1.0e-04  TaskW=[bin=1.00 def=1.00 prod=1.00]  [FROZEN]

  Train → Bin=99.0%  Def=77.2%  F1=0.773  Prod=97.0%
  Val   → Bin=98.0%  Def=80.1%  F1=0.758  Prod=99.9%

  Class                       P       R      F1
  ----------------------------------------
  contamination           0.824   0.848   0.836
  cut                     0.929   0.765   0.839
  deformation             0.714   0.333   0.455
  fracture                0.850   0.791   0.819
  hole_void               0.944   0.944   0.944
  minor_defect            0.527   0.929   0.672
  scratch                 0.824   0.483   0.609
  surface_quality         0.946   0.841   0.891
  ----------------------------------------
  MACRO                   0.820   0.742   0.758

  ✅ NEW BEST  F1=0.758 → saved EdgeNet_V2_KD.pth

Epoch [2/100]  LR=1.0e-04  TaskW=[bin=1.00 def=1.00 prod=1.00]  [FROZEN]

  Train → Bin=99.2%  Def=77.6%  F1=0.777  Prod=97.1%
  Val   → Bin=97.9%  Def=79.4%  F1=0.733  Prod=99.8%

  Class             

In [18]:
# ======================================================================
# KD_CELL_6 — FINAL TEST SET EVALUATION
# Evaluates both EdgeNet-V2 (original) and EdgeNet-V2+KD on test set
# Produces the thesis comparison table
# ======================================================================

from sklearn.metrics import (precision_recall_fscore_support,
                              confusion_matrix, classification_report)
import numpy as np
import torch

# ── Helper: full evaluation on any loader ─────────────────────────────
def evaluate_on_test(model, loader, model_name="Model"):
    model.eval()
    all_pred, all_true = [], []
    bin_c = prod_c = tot = def_c = def_tot = 0

    with torch.no_grad():
        for batch in loader:
            imgs, bin_lbl, def_lbl, prod_lbl, _ = batch
            imgs     = imgs.to(device)
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)

            sb, sd, sp = model(imgs)

            bin_c  += ((torch.sigmoid(sb.squeeze()) > 0.5).long()
                       == bin_lbl).sum().item()
            prod_c += (sp.argmax(1) == prod_lbl).sum().item()
            tot    += imgs.size(0)

            known = def_lbl != -1
            if known.sum() > 0:
                preds = sd[known].argmax(1)
                def_c   += (preds == def_lbl[known]).sum().item()
                def_tot += known.sum().item()
                all_pred.extend(preds.cpu().tolist())
                all_true.extend(def_lbl[known].cpu().tolist())

    bin_acc  = 100. * bin_c  / max(tot,     1)
    def_acc  = 100. * def_c  / max(def_tot, 1)
    prod_acc = 100. * prod_c / max(tot,     1)

    p, r, f, s = precision_recall_fscore_support(
        all_true, all_pred,
        labels=list(range(NUM_DEFECT_TYPES)),
        zero_division=0)
    f1_macro = float(f.mean())

    cm = confusion_matrix(all_true, all_pred,
                          labels=list(range(NUM_DEFECT_TYPES)))

    return {
        'name':      model_name,
        'bin_acc':   bin_acc,
        'def_acc':   def_acc,
        'prod_acc':  prod_acc,
        'f1_macro':  f1_macro,
        'precision': p,
        'recall':    r,
        'f1':        f,
        'support':   s,
        'cm':        cm,
        'all_pred':  all_pred,
        'all_true':  all_true,
    }


# ══════════════════════════════════════════════════════════════════════
# 1. Load test dataloader
# ══════════════════════════════════════════════════════════════════════
from torch.utils.data import DataLoader

test_3head_loader = DataLoader(
    test_3head_ds,
    batch_size  = 32,
    shuffle     = False,
    num_workers = 4,
    pin_memory  = True,
)
print(f"Test batches : {len(test_3head_loader)}  "
      f"({len(test_3head_ds):,} samples)")


# ══════════════════════════════════════════════════════════════════════
# 2. Evaluate EdgeNet-V2 (original, no KD)
# ══════════════════════════════════════════════════════════════════════
print("\nLoading original EdgeNet-V2 checkpoint...")
ck_orig = torch.load(
    '/home/sufi/training_results/models/EdgeNet_V2.pth',
    map_location=device, weights_only=False)
model.load_state_dict(ck_orig['model_state_dict'])
res_orig = evaluate_on_test(model, test_3head_loader, "EdgeNet-V2")
print("✅ EdgeNet-V2 evaluation complete")


# ══════════════════════════════════════════════════════════════════════
# 3. Evaluate EdgeNet-V2+KD (best KD checkpoint)
# ══════════════════════════════════════════════════════════════════════
print("Loading EdgeNet-V2+KD checkpoint...")
ck_kd = torch.load(
    '/home/sufi/training_results/models/EdgeNet_V2_KD.pth',
    map_location=device, weights_only=False)
model.load_state_dict(ck_kd['model_state_dict'])
res_kd = evaluate_on_test(model, test_3head_loader, "EdgeNet-V2+KD")
print(f"✅ EdgeNet-V2+KD evaluation complete  "
      f"(KD best epoch: {ck_kd['epoch']})")


# ══════════════════════════════════════════════════════════════════════
# 4. Evaluate Teacher on test set
# ══════════════════════════════════════════════════════════════════════
print("Evaluating Teacher on test set...")
res_teacher = evaluate_on_test(teacher, test_3head_loader, "EfficientNet-B0")
print("✅ Teacher evaluation complete")


# ══════════════════════════════════════════════════════════════════════
# 5. Print full thesis comparison table
# ══════════════════════════════════════════════════════════════════════
print()
print("=" * 70)
print("FINAL TEST SET RESULTS — THESIS COMPARISON TABLE")
print("=" * 70)

BASELINES = [
    ("MobileNetV3-Small",  "79.05",  "—",     "—"),
    ("ResNet-18",          "84.46",  "—",     "—"),
    ("ResNet-50",          "90.88",  "—",     "—"),
    ("MobileNetV3-Large",  "91.22",  "—",     "—"),
]

header = (f"  {'Model':<26} {'Params':>7}  "
          f"{'Binary':>8}  {'Defect':>8}  {'F1':>7}  {'Product':>9}")
divider = "  " + "-" * 68
print(header)
print(divider)

# Baselines (from saved results — no test re-run needed)
for name, defect, f1, prod in BASELINES:
    print(f"  {name:<26} {'—':>7}  {'—':>8}  "
          f"{defect:>7}%  {f1:>7}  {prod:>8}")

print(divider)

# Teacher
print(f"  {'EfficientNet-B0 (Teacher)':<26} {'5.00M':>7}  "
      f"{res_teacher['bin_acc']:>7.2f}%  "
      f"{res_teacher['def_acc']:>7.2f}%  "
      f"{res_teacher['f1_macro']:>7.3f}  "
      f"{res_teacher['prod_acc']:>8.2f}%  ← Teacher")

print(divider)

# EdgeNet-V2 original
print(f"  {'EdgeNet-V2':<26} {'3.89M':>7}  "
      f"{res_orig['bin_acc']:>7.2f}%  "
      f"{res_orig['def_acc']:>7.2f}%  "
      f"{res_orig['f1_macro']:>7.3f}  "
      f"{res_orig['prod_acc']:>8.2f}%")

# EdgeNet-V2+KD — highlight best
print(f"  {'EdgeNet-V2+KD (Ours)':<26} {'3.89M':>7}  "
      f"{res_kd['bin_acc']:>7.2f}%  "
      f"{res_kd['def_acc']:>7.2f}%  "
      f"{res_kd['f1_macro']:>7.3f}  "
      f"{res_kd['prod_acc']:>8.2f}%  ← Best")

print("=" * 70)


# ══════════════════════════════════════════════════════════════════════
# 6. Per-class breakdown: Original vs KD
# ══════════════════════════════════════════════════════════════════════
print()
print("=" * 70)
print("PER-CLASS F1 — EdgeNet-V2  vs  EdgeNet-V2+KD  (Test Set)")
print("=" * 70)
print(f"  {'Class':<22}  {'Original':>10}  {'KD':>10}  {'Gain':>8}  {'Support':>8}")
print("  " + "-" * 60)

for i, name in enumerate(SEMANTIC_GROUPS):
    f_orig = res_orig['f1'][i]
    f_kd   = res_kd['f1'][i]
    gain   = f_kd - f_orig
    sup    = int(res_kd['support'][i])
    flag   = "  ▲" if gain > 0.02 else ("  ▼" if gain < -0.02 else "")
    print(f"  {name:<22}  {f_orig:>10.3f}  {f_kd:>10.3f}  "
          f"{gain:>+8.3f}  {sup:>8}{flag}")

print("  " + "-" * 60)
print(f"  {'MACRO':<22}  {res_orig['f1_macro']:>10.3f}  "
      f"{res_kd['f1_macro']:>10.3f}  "
      f"{res_kd['f1_macro'] - res_orig['f1_macro']:>+8.3f}")

print("=" * 70)


# ══════════════════════════════════════════════════════════════════════
# 7. KD gain summary
# ══════════════════════════════════════════════════════════════════════
print()
print("=" * 70)
print("KD KNOWLEDGE TRANSFER SUMMARY")
print("=" * 70)
print(f"  Teacher defect acc   : {res_teacher['def_acc']:.2f}%")
print(f"  Student before KD    : {res_orig['def_acc']:.2f}%")
print(f"  Student after  KD    : {res_kd['def_acc']:.2f}%")
gap_closed = res_kd['def_acc'] - res_orig['def_acc']
gap_total  = res_teacher['def_acc'] - res_orig['def_acc']
pct_closed = 100. * gap_closed / max(gap_total, 1e-6)
print(f"  Gap closed by KD     : +{gap_closed:.2f}%  "
      f"({pct_closed:.1f}% of teacher gap)")
print(f"  Params (student)     : 3.89M  (teacher: 5.00M, "
      f"savings: {(1 - 3.89/5.00)*100:.1f}%)")
print(f"  KD best epoch        : {ck_kd['epoch']}")
print("=" * 70)
print("\n✅ KD_CELL_6 DONE — Thesis evaluation complete")

Test batches : 53  (1,668 samples)

Loading original EdgeNet-V2 checkpoint...
✅ EdgeNet-V2 evaluation complete
Loading EdgeNet-V2+KD checkpoint...
✅ EdgeNet-V2+KD evaluation complete  (KD best epoch: 88)
Evaluating Teacher on test set...
✅ Teacher evaluation complete

FINAL TEST SET RESULTS — THESIS COMPARISON TABLE
  Model                       Params    Binary    Defect       F1    Product
  --------------------------------------------------------------------
  MobileNetV3-Small                —         —    79.05%        —         —
  ResNet-18                        —         —    84.46%        —         —
  ResNet-50                        —         —    90.88%        —         —
  MobileNetV3-Large                —         —    91.22%        —         —
  --------------------------------------------------------------------
  EfficientNet-B0 (Teacher)    5.00M    98.68%    91.22%    0.897     99.76%  ← Teacher
  --------------------------------------------------------------------
